In [ ]:
import ast
import h5py
import numpy as np
import matplotlib.pyplot as plt


def plot_extreme_chirp_mass_waveforms(file_path, xlim=None):
    """
    Loads a Dingo HDF5 waveform dataset, finds the waveforms with the
    lowest and highest chirp_mass, and plots their h_plus and h_cross
    polarizations in the time domain.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 waveform dataset.
    xlim : tuple of (float, float), optional
        Time window to zoom into, e.g., (-2.0, 0.2). Defaults to full duration.
    """
    with h5py.File(file_path, "r") as f:
        # 1. Parse settings from root attributes
        raw_settings = f.attrs["settings"]
        if isinstance(raw_settings, bytes):
            raw_settings = raw_settings.decode("utf-8")
        settings = (
            ast.literal_eval(raw_settings)
            if isinstance(raw_settings, str)
            else dict(raw_settings)
        )

        # 2. Find indices of lowest and highest chirp_mass
        chirp_masses = f["parameters"]["chirp_mass"][:]
        min_idx = int(np.argmin(chirp_masses))
        max_idx = int(np.argmax(chirp_masses))

        events = [
            ("Lowest Chirp Mass", min_idx),
            ("Highest Chirp Mass", max_idx),
        ]

        # Extract domain parameters
        domain = settings.get("domain", settings.get("domain_dict", settings))
        f_min = float(domain.get("f_min", 2.0))
        f_max = float(domain.get("f_max", 1024.0))
        delta_f = float(domain.get("delta_f", 1.0 / 64.0))
        sampling_rate = float(domain.get("sampling_rate", 2.0 * f_max))

        duration = 1.0 / delta_f
        n_time_samples = int(round(duration * sampling_rate))
        n_pad_left = int(round(f_min / delta_f))
        expected_bins = n_time_samples // 2 + 1
        time = np.linspace(-duration / 2, duration / 2, n_time_samples, endpoint=False)

        # 3. Load and transform both waveforms
        plot_data = []
        field_names = f["parameters"].dtype.names or ()

        for label, idx in events:
            hp_raw = f["polarizations/h_plus"][idx]
            hc_raw = f["polarizations/h_cross"][idx]

            hp_fd = hp_raw["r"] + 1j * hp_raw["i"] if hp_raw.dtype.names else hp_raw
            hc_fd = hc_raw["r"] + 1j * hc_raw["i"] if hc_raw.dtype.names else hc_raw

            n_pad_right = max(0, expected_bins - n_pad_left - len(hp_fd))
            hp_fd_full = np.pad(hp_fd, (n_pad_left, n_pad_right))
            hc_fd_full = np.pad(hc_fd, (n_pad_left, n_pad_right))

            hp_td = np.fft.fftshift(
                np.fft.irfft(hp_fd_full, n=n_time_samples) * sampling_rate
            )
            hc_td = np.fft.fftshift(
                np.fft.irfft(hc_fd_full, n=n_time_samples) * sampling_rate
            )

            param_row = f["parameters"][idx]
            params = {
                key: float(param_row[key])
                for key in ["chirp_mass", "mass_ratio", "luminosity_distance"]
                if key in field_names
            }
            plot_data.append((label, idx, hp_td, hc_td, params))

    # 4. Plot in a 2x2 grid (Top row: h_plus, Bottom row: h_cross)
    fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex=True)

    for col, (label, idx, hp_td, hc_td, params) in enumerate(plot_data):
        ax_plus = axes[0, col]
        ax_cross = axes[1, col]

        title = (
            f"{label} (Event {idx})\n"
            f"$\\mathcal{{M}}$ = {params.get('chirp_mass', 0):.2f} $M_\\odot$, "
            f"$q$ = {params.get('mass_ratio', 0):.2f}, "
            f"$d_L$ = {params.get('luminosity_distance', 0):.0f} Mpc"
        )

        ax_plus.plot(time, hp_td, color="#1f77b4", lw=1.0)
        ax_plus.set_title(title)
        ax_plus.grid(True, alpha=0.3)

        ax_cross.plot(time, hc_td, color="#ff7f0e", lw=1.0)
        ax_cross.set_xlabel("Time relative to merger (s)")
        ax_cross.grid(True, alpha=0.3)

        if xlim is not None:
            ax_plus.set_xlim(xlim)

    axes[0, 0].set_ylabel(r"$h_+(t)$ [Strain]")
    axes[1, 0].set_ylabel(r"$h_\times(t)$ [Strain]")

    plt.tight_layout()
    plt.show()



# 6-256 hz

In [ ]:

plot_extreme_chirp_mass_waveforms("training_data/waveform_dataset2.hdf5")
plot_extreme_chirp_mass_waveforms("training_data/waveform_dataset1.hdf5")

# 6-512hz


In [ ]:
plot_extreme_chirp_mass_waveforms("training_data/waveform_dataset3.hdf5")


# 6-1024 hz

In [ ]:
plot_extreme_chirp_mass_waveforms("training_data/waveform_dataset4.hdf5")

In [ ]:
import ast
import h5py
import numpy as np
import matplotlib.pyplot as plt


def plot_extreme_chirp_mass_waveforms_fd(
    file_path, characteristic_strain=False, xlim=None
):
    """
    Loads a Dingo HDF5 waveform dataset, finds the waveforms with the
    lowest and highest chirp_mass, and plots their frequency-domain strain
    amplitude (|h(f)| or characteristic strain 2*f*|h(f)|) on a log-log scale.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 waveform dataset.
    characteristic_strain : bool, optional
        If True, plots dimensionless characteristic strain h_c(f) = 2 * f * |h(f)|.
        If False (default), plots strain spectral density |h(f)| [1/Hz].
    xlim : tuple of (float, float), optional
        Frequency range (f_low, f_high) in Hz to display.
    """
    with h5py.File(file_path, "r") as f:
        # 1. Parse settings from root attributes
        raw_settings = f.attrs["settings"]
        if isinstance(raw_settings, bytes):
            raw_settings = raw_settings.decode("utf-8")
        settings = (
            ast.literal_eval(raw_settings)
            if isinstance(raw_settings, str)
            else dict(raw_settings)
        )

        # 2. Find indices of lowest and highest chirp_mass
        chirp_masses = f["parameters"]["chirp_mass"][:]
        min_idx = int(np.argmin(chirp_masses))
        max_idx = int(np.argmax(chirp_masses))

        events = [
            ("Lowest Chirp Mass", min_idx),
            ("Highest Chirp Mass", max_idx),
        ]

        # Extract domain parameters
        domain = settings.get("domain", settings.get("domain_dict", settings))
        f_min = float(domain.get("f_min", 2.0))
        f_max = float(domain.get("f_max", 1024.0))
        delta_f = float(domain.get("delta_f", 1.0 / 64.0))

        # 3. Load complex frequency-domain polarizations
        plot_data = []
        field_names = f["parameters"].dtype.names or ()

        for label, idx in events:
            hp_raw = f["polarizations/h_plus"][idx]
            hc_raw = f["polarizations/h_cross"][idx]

            hp_fd = hp_raw["r"] + 1j * hp_raw["i"] if hp_raw.dtype.names else hp_raw
            hc_fd = hc_raw["r"] + 1j * hc_raw["i"] if hc_raw.dtype.names else hc_raw

            # Build frequency array matching the stored bins [f_min, f_min + delta_f, ...]
            freqs = f_min + np.arange(len(hp_fd)) * delta_f

            # Compute amplitude (or characteristic strain)
            hp_amp = np.abs(hp_fd)
            hc_amp = np.abs(hc_fd)

            if characteristic_strain:
                hp_amp = 2.0 * freqs * hp_amp
                hc_amp = 2.0 * freqs * hc_amp

            param_row = f["parameters"][idx]
            params = {
                key: float(param_row[key])
                for key in ["chirp_mass", "mass_ratio", "luminosity_distance"]
                if key in field_names
            }
            plot_data.append((label, idx, freqs, hp_amp, hc_amp, params))

    # 4. Plot in a 2x2 log-log grid (Top row: h_plus, Bottom row: h_cross)
    fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex=True, sharey="row")

    ylabel_plus = (
        r"$h_{c,+}(f) = 2f|\tilde{h}_+(f)|$"
        if characteristic_strain
        else r"$|\tilde{h}_+(f)|$ [$\mathrm{Hz}^{-1}$]"
    )
    ylabel_cross = (
        r"$h_{c,\times}(f) = 2f|\tilde{h}_\times(f)|$"
        if characteristic_strain
        else r"$|\tilde{h}_\times(f)|$ [$\mathrm{Hz}^{-1}$]"
    )

    for col, (label, idx, freqs, hp_amp, hc_amp, params) in enumerate(plot_data):
        ax_plus = axes[0, col]
        ax_cross = axes[1, col]

        # Mask out exact zeros so log-log scaling stays clean
        mask_p = hp_amp > 0
        mask_c = hc_amp > 0

        title = (
            f"{label} (Event {idx})\n"
            f"$\\mathcal{{M}}$ = {params.get('chirp_mass', 0):.2f} $M_\\odot$, "
            f"$q$ = {params.get('mass_ratio', 0):.2f}, "
            f"$d_L$ = {params.get('luminosity_distance', 0):.0f} Mpc"
        )

        ax_plus.loglog(freqs[mask_p], hp_amp[mask_p], color="#1f77b4", lw=1.2)
        ax_plus.set_title(title)
        ax_plus.grid(True, which="both", alpha=0.3)

        ax_cross.loglog(freqs[mask_c], hc_amp[mask_c], color="#ff7f0e", lw=1.2)
        ax_cross.set_xlabel("Frequency $f$ [Hz]")
        ax_cross.grid(True, which="both", alpha=0.3)

        if xlim is not None:
            ax_plus.set_xlim(xlim)
        else:
            ax_plus.set_xlim(f_min, f_max)

    axes[0, 0].set_ylabel(ylabel_plus)
    axes[1, 0].set_ylabel(ylabel_cross)

    plt.tight_layout()
    plt.show()



In [ ]:
plot_extreme_chirp_mass_waveforms_fd("training_data/waveform_dataset.hdf5")
plot_extreme_chirp_mass_waveforms_fd("training_data/waveform_dataset1.hdf5")
plot_extreme_chirp_mass_waveforms_fd("training_data/waveform_dataset2.hdf5")



In [ ]:
plot_extreme_chirp_mass_waveforms_fd("training_data/waveform_dataset5.hdf5")

In [ ]:
plot_extreme_chirp_mass_waveforms_fd("training_data/waveform_dataset6.hdf5")
plot_extreme_chirp_mass_waveforms("training_data/waveform_dataset6.hdf5")

In [ ]:
plot_extreme_chirp_mass_waveforms_fd("training_data/waveform_dataset7.hdf5")
plot_extreme_chirp_mass_waveforms("training_data/waveform_dataset7.hdf5")

In [ ]:
import ast
import math
import h5py
import numpy as np
import matplotlib.pyplot as plt


def plot_sorted_chirp_mass_waveforms_td(
    file_path, n_cols=4, max_waveforms=None, xlim=None
):
    """
    Loads a Dingo HDF5 waveform dataset, sorts all waveforms in order of
    increasing chirp_mass, converts them to the time domain, and plots
    h_plus and h_cross in a grid.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 waveform dataset.
    n_cols : int, optional
        Number of columns in the subplot grid (default is 4).
    max_waveforms : int, optional
        Optional cap on the number of sorted waveforms to plot.
        If None (default), plots all waveforms in the dataset.
    xlim : tuple of (float, float), optional
        Time window (t_start, t_end) in seconds to zoom into, e.g., (-2.0, 0.2).
    """
    with h5py.File(file_path, "r") as f:
        # 1. Parse settings from root attributes
        raw_settings = f.attrs["settings"]
        if isinstance(raw_settings, bytes):
            raw_settings = raw_settings.decode("utf-8")
        settings = (
            ast.literal_eval(raw_settings)
            if isinstance(raw_settings, str)
            else dict(raw_settings)
        )

        # 2. Sort all indices by increasing chirp_mass
        chirp_masses = f["parameters"]["chirp_mass"][:]
        sorted_indices = np.argsort(chirp_masses)
        if max_waveforms is not None:
            sorted_indices = sorted_indices[:max_waveforms]

        # Extract domain parameters
        domain = settings.get("domain", settings.get("domain_dict", settings))
        f_min = float(domain.get("f_min", 2.0))
        f_max = float(domain.get("f_max", 1024.0))
        delta_f = float(domain.get("delta_f", 1.0 / 64.0))
        sampling_rate = float(domain.get("sampling_rate", 2.0 * f_max))

        duration = 1.0 / delta_f
        n_time_samples = int(round(duration * sampling_rate))
        n_pad_left = int(round(f_min / delta_f))
        expected_bins = n_time_samples // 2 + 1
        time = np.linspace(-duration / 2, duration / 2, n_time_samples, endpoint=False)

        # 3. Load and transform all sorted waveforms
        plot_data = []
        field_names = f["parameters"].dtype.names or ()

        for idx in sorted_indices:
            idx = int(idx)
            hp_raw = f["polarizations/h_plus"][idx]
            hc_raw = f["polarizations/h_cross"][idx]

            hp_fd = hp_raw["r"] + 1j * hp_raw["i"] if hp_raw.dtype.names else hp_raw
            hc_fd = hc_raw["r"] + 1j * hc_raw["i"] if hc_raw.dtype.names else hc_raw

            n_pad_right = max(0, expected_bins - n_pad_left - len(hp_fd))
            hp_fd_full = np.pad(hp_fd, (n_pad_left, n_pad_right))
            hc_fd_full = np.pad(hc_fd, (n_pad_left, n_pad_right))

            hp_td = np.fft.fftshift(
                np.fft.irfft(hp_fd_full, n=n_time_samples) * sampling_rate
            )
            hc_td = np.fft.fftshift(
                np.fft.irfft(hc_fd_full, n=n_time_samples) * sampling_rate
            )

            param_row = f["parameters"][idx]
            params = {
                key: float(param_row[key])
                for key in ["chirp_mass", "mass_ratio", "luminosity_distance"]
                if key in field_names
            }
            plot_data.append((idx, hp_td, hc_td, params))

    # 4. Plot in a grid ordered left-to-right, top-to-bottom by increasing chirp mass
    n_events = len(plot_data)
    n_cols = min(n_cols, n_events)
    n_rows = math.ceil(n_events / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.2 * n_cols, 2.8 * n_rows),
        sharex=True,
        squeeze=False,
    )

    for plot_idx, (idx, hp_td, hc_td, params) in enumerate(plot_data):
        row = plot_idx // n_cols
        col = plot_idx % n_cols
        ax = axes[row, col]

        ax.plot(time, hp_td, color="#1f77b4", lw=0.9, label=r"$h_+(t)$")
        ax.plot(time, hc_td, color="#ff7f0e", lw=0.9, alpha=0.85, label=r"$h_\times(t)$")

        title = (
            f"#{plot_idx + 1} (Idx {idx}) | "
            f"$\\mathcal{{M}}$={params.get('chirp_mass', 0):.2f} $M_\\odot$, "
            f"$q$={params.get('mass_ratio', 0):.2f}"
        )
        ax.set_title(title, fontsize=9)
        ax.grid(True, alpha=0.3)

        if xlim is not None:
            ax.set_xlim(xlim)

        if col == 0:
            ax.set_ylabel("Strain")
        if row == n_rows - 1:
            ax.set_xlabel("Time relative to merger (s)")

    # Add a single shared legend at the top and hide any unused trailing subplots
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=11)

    for empty_idx in range(n_events, n_rows * n_cols):
        axes[empty_idx // n_cols, empty_idx % n_cols].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()





In [ ]:
# Example usage:
plot_sorted_chirp_mass_waveforms_td("training_data/waveform_dataset8.hdf5")

In [ ]:
import ast
import math
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal.windows import tukey


def plot_sorted_chirp_mass_waveforms_td(
    file_path, n_cols=4, max_waveforms=None, xlim=None, tukey_alpha=0.4, merger_pos=0.75
):
    """
    Loads a Dingo HDF5 waveform dataset, sorts all waveforms in order of
    increasing chirp_mass, converts them to the time domain, shifts the merger
    to `merger_pos` of the duration (default 0.75 -> 6s for an 8s window),
    applies a Tukey window, and plots h_plus and h_cross in a grid.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 waveform dataset.
    n_cols : int, optional
        Number of columns in the subplot grid (default is 4).
    max_waveforms : int, optional
        Optional cap on the number of sorted waveforms to plot.
    xlim : tuple of (float, float), optional
        Time window (t_start, t_end) in seconds to zoom into, e.g., (4.0, 6.5).
    tukey_alpha : float, optional
        Shape parameter alpha of the Tukey window (default is 0.4).
    merger_pos : float, optional
        Fractional position of the merger in the time window (default is 0.75,
        which corresponds to 6.0 s in an 8.0 s duration).
    """
    with h5py.File(file_path, "r") as f:
        # 1. Parse settings from root attributes
        raw_settings = f.attrs["settings"]
        if isinstance(raw_settings, bytes):
            raw_settings = raw_settings.decode("utf-8")
        settings = (
            ast.literal_eval(raw_settings)
            if isinstance(raw_settings, str)
            else dict(raw_settings)
        )

        # 2. Sort all indices by increasing chirp_mass
        chirp_masses = f["parameters"]["chirp_mass"][:]
        sorted_indices = np.argsort(chirp_masses)
        if max_waveforms is not None:
            sorted_indices = sorted_indices[:max_waveforms]

        # Extract domain parameters
        domain = settings.get("domain", settings.get("domain_dict", settings))
        f_min = float(domain.get("f_min", 2.0))
        f_max = float(domain.get("f_max", 1024.0))
        delta_f = float(domain.get("delta_f", 1.0 / 64.0))
        sampling_rate = float(domain.get("sampling_rate", 2.0 * f_max))

        duration = 1.0 / delta_f
        n_time_samples = int(round(duration * sampling_rate))
        n_pad_left = int(round(f_min / delta_f))
        expected_bins = n_time_samples // 2 + 1

        # Time axis from 0 to duration (e.g., 0 to 8 s), placing merger at 6 s
        time = np.arange(n_time_samples) / sampling_rate
        shift_samples = int(round(merger_pos * n_time_samples))
        window = tukey(n_time_samples, alpha=tukey_alpha)

        # 3. Load, transform, shift, and window all sorted waveforms
        plot_data = []
        field_names = f["parameters"].dtype.names or ()

        for idx in sorted_indices:
            idx = int(idx)
            hp_raw = f["polarizations/h_plus"][idx]
            hc_raw = f["polarizations/h_cross"][idx]

            hp_fd = hp_raw["r"] + 1j * hp_raw["i"] if hp_raw.dtype.names else hp_raw
            hc_fd = hc_raw["r"] + 1j * hc_raw["i"] if hc_raw.dtype.names else hc_raw

            n_pad_right = max(0, expected_bins - n_pad_left - len(hp_fd))
            hp_fd_full = np.pad(hp_fd, (n_pad_left, n_pad_right))
            hc_fd_full = np.pad(hc_fd, (n_pad_left, n_pad_right))

            # Inverse RFFT -> Roll merger to 3/4 (6s) -> Apply 0.4 Tukey window
            hp_td = np.roll(
                np.fft.irfft(hp_fd_full, n=n_time_samples) * sampling_rate,
                shift_samples,
            ) * window
            hc_td = np.roll(
                np.fft.irfft(hc_fd_full, n=n_time_samples) * sampling_rate,
                shift_samples,
            ) * window

            param_row = f["parameters"][idx]
            params = {
                key: float(param_row[key])
                for key in ["chirp_mass", "mass_ratio", "luminosity_distance"]
                if key in field_names
            }
            plot_data.append((idx, hp_td, hc_td, params))

    # 4. Plot in a grid ordered left-to-right, top-to-bottom by increasing chirp mass
    n_events = len(plot_data)
    n_cols = min(n_cols, n_events)
    n_rows = math.ceil(n_events / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.2 * n_cols, 2.8 * n_rows),
        sharex=True,
        squeeze=False,
    )

    for plot_idx, (idx, hp_td, hc_td, params) in enumerate(plot_data):
        row = plot_idx // n_cols
        col = plot_idx % n_cols
        ax = axes[row, col]

        ax.plot(time, hp_td, color="#1f77b4", lw=0.9, label=r"$h_+(t)$")
        ax.plot(time, hc_td, color="#ff7f0e", lw=0.9, alpha=0.85, label=r"$h_\times(t)$")

        title = (
            f"#{plot_idx + 1} (Idx {idx}) | "
            f"$\\mathcal{{M}}$={params.get('chirp_mass', 0):.2f} $M_\\odot$, "
            f"$q$={params.get('mass_ratio', 0):.2f}"
        )
        ax.set_title(title, fontsize=9)
        ax.grid(True, alpha=0.3)

        if xlim is not None:
            ax.set_xlim(xlim)

        if col == 0:
            ax.set_ylabel("Strain")
        if row == n_rows - 1:
            ax.set_xlabel("Time (s)")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=11)

    for empty_idx in range(n_events, n_rows * n_cols):
        axes[empty_idx // n_cols, empty_idx % n_cols].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()





In [ ]:
# Example usage:
plot_sorted_chirp_mass_waveforms_td("training_data/waveform_dataset8.hdf5")